## Example 1: Read an HDF5 file h5py wrote, without libhdf5

In [ ]:
import importlib.metadata
import os
import tempfile

import h5py
import numpy as np
import pyfive

# Example 1: write with h5py (libhdf5), read back with pyfive (pure Python)
assert importlib.metadata.version("pyfive") == "1.3.1"
path = os.path.join(tempfile.mkdtemp(), "data.h5")
with h5py.File(path, "w") as f:
    f["grid"] = np.arange(12, dtype="f8").reshape(3, 4)
with pyfive.File(path) as f:
    grid = f["grid"][:]
assert grid.shape == (3, 4) and grid.dtype == np.float64 and grid[2, 3] == 11.0, grid
print("Example 1 OK:", grid.sum())

## Example 2: Groups and attributes

In [ ]:
import os
import tempfile

import h5py
import pyfive

# Example 2: nested groups and attributes of several types are read back as written
path = os.path.join(tempfile.mkdtemp(), "meta.h5")
with h5py.File(path, "w") as f:
    run = f.create_group("experiments/run1")
    run.attrs["temperature"] = 21.5
    run.attrs["label"] = "baseline"
    run["counts"] = [1, 2, 3]
with pyfive.File(path) as f:
    run = f["experiments"]["run1"]
    assert run.attrs["temperature"] == 21.5 and run.attrs["label"] in ("baseline", b"baseline"), dict(run.attrs)
    assert list(run["counts"][:]) == [1, 2, 3] and sorted(f.keys()) == ["experiments"]
print("Example 2 OK:", dict(run.attrs))

## Example 3: Chunked, compressed datasets read in slices

In [ ]:
import os
import tempfile

import h5py
import numpy as np
import pyfive

# Example 3: a gzip+shuffle chunked dataset; a slice reads only the chunks it needs
path = os.path.join(tempfile.mkdtemp(), "chunked.h5")
data = np.random.default_rng(0).integers(0, 1000, size=(200, 300), dtype="i4")
with h5py.File(path, "w") as f:
    f.create_dataset("x", data=data, chunks=(50, 60), compression="gzip", shuffle=True)
with pyfive.File(path) as f:
    ds = f["x"]
    assert ds.shape == (200, 300) and ds.chunks == (50, 60), (ds.shape, ds.chunks)
    assert np.array_equal(ds[120:180, 30:90], data[120:180, 30:90])
print("Example 3 OK:", ds.dtype, ds.compression)

## Example 4: Open a file-like object (the fsspec route)

In [ ]:
import io
import os
import tempfile

import fsspec
import h5py
import pyfive

# Example 4: pyfive reads any seekable file object, e.g. one fsspec opens (local here, S3/HTTP the same way)
path = os.path.join(tempfile.mkdtemp(), "remote.h5")
with h5py.File(path, "w") as f:
    f["v"] = [10, 20, 30]
with fsspec.open(f"file://{path}", "rb") as handle:
    with pyfive.File(handle) as f:
        assert list(f["v"][:]) == [10, 20, 30]
with pyfive.File(io.BytesIO(open(path, "rb").read())) as f:
    assert f["v"][1] == 20
print("Example 4 OK")

## Example 5: A netCDF-4 file written by h5netcdf

In [ ]:
import os
import tempfile

import h5netcdf
import numpy as np
import pyfive

# Example 5: netCDF-4 is HDF5 underneath; pyfive sees its variables, dimensions-as-datasets and attributes
path = os.path.join(tempfile.mkdtemp(), "climate.nc")
with h5netcdf.File(path, "w") as f:
    f.dimensions = {"time": 4}
    var = f.create_variable("temp", ("time",), float)
    var[:] = np.array([1.5, 2.5, 3.5, 4.5])
    var.attrs["units"] = "K"
with pyfive.File(path) as f:
    temp = f["temp"]
    assert np.allclose(temp[:], [1.5, 2.5, 3.5, 4.5]) and temp.attrs["units"] in ("K", b"K"), temp.attrs
print("Example 5 OK: temp", temp.attrs["units"])